In [ ]:
from dataclasses import dataclass, replace, asdict
from typing import get_type_hints
from typing import List, Dict
from pathlib import Path

import csv
import numpy as np
import pickle
from RRAM import (
    CurrentSolver,
    Representate,
    exceptions,
    Percolation,
    Temperature,
    ElectricField,
    Generation
)

In [3]:
def read_csv_to_dic(cvs_path: str) -> List[Dict[str, str]]:
    """
    Reads a CSV file and returns its contents as a list of dictionaries.
    Args:
        cvs_path (str): The name of the CSV file to read.

    Returns:
        list: A list of dictionaries, where each dictionary represents a row in the CSV file.
    """
    with open(cvs_path, mode="r") as archivo:
        reader = csv.DictReader(archivo)
        data = [row for row in reader]
    return data


In [4]:
@dataclass
class SimulationParameters:
    device_size: float
    atom_size: float
    x_size: int
    y_size: int
    num_trampas: int
    init_simulation_time: float
    total_simulation_time: float
    num_pasos: int
    paso_guardar: int
    voltaje_final_reset: float
    voltaje_final_set: float
    initial_voltaje: float
    initial_current: float
    initial_elec_field: float
    initial_temperatura: float
    T_0: float

    @staticmethod
    def from_dict(d: dict):
        # Usar get_type_hints para asegurar que obtienes tipos reales
        field_types = get_type_hints(SimulationParameters)

        mapping = {
            "voltaje_final_reset": "voltaje_final",
            "T_0": "init_temp",
            "initial_temperatura": "init_temp",
        }

        kwargs = {}
        for k in field_types:
            src = mapping.get(k, k)

            # Verificar que la clave existe en el diccionario
            if src not in d:
                raise KeyError(f"La clave '{src}' no existe en el diccionario")

            # Debug: ver qué tipo y valor estamos procesando
            # print(f"Campo: {k}, Tipo: {field_types[k]}, Fuente: {src}, Valor: {d[src]}")

            try:
                kwargs[k] = field_types[k](d[src])
            except Exception as e:
                print(f"Error al convertir {k}: {e}")
                print(
                    f"Tipo esperado: {field_types[k]}, tipo real: {type(field_types[k])}"
                )
                raise

        return SimulationParameters(**kwargs)
    

In [9]:
@dataclass(frozen=True)

class SimulationConstants:
    vibration_frequency: float
    migration_energy: float
    drift_coefficient: float
    cte_red: float
    recom_enchancement_factor: float
    decaimiento_concentracion: float
    activation_energy: float
    gamma: float
    ohm_resistence: float
    pb_metal_insul: float
    permitividad_relativa: float
    I_0: float
    r_termica_percola: float
    r_termica_no_percola: float
    factor_generacion: float
    recombination_energy: float

    @staticmethod
    def from_dict(d: dict):
        # Obtiene los tipos reales de SimulationConstants, no de SimulationParameters
        field_types = get_type_hints(SimulationConstants)

        # Alias si es necesario; si no, mapa vacío sirve
        mapping = {}

        kwargs = {}
        for k in field_types:
            src = mapping.get(k, k)

            if src not in d:
                raise KeyError(f"La clave '{src}' no existe en el diccionario")

            try:
                kwargs[k] = field_types[k](d[src])
            except Exception as e:
                print(f"Error al convertir {k} con valor {d[src]}: {e}")
                raise

        return SimulationConstants(**kwargs)
    
        
    def update_gamma(self, nuevo_valor_gamma: float):
        return replace(self, gamma=nuevo_valor_gamma)


In [ ]:
sim_parmtrs = read_csv_to_dic("Init_data/simulation_parameters.csv")
params = SimulationParameters.from_dict(sim_parmtrs[0])

sim_cte = read_csv_to_dic("Init_data/simulation_constants.csv")
ctes = SimulationConstants.from_dict(sim_cte[0])

SimulationConstants(vibration_frequency=10000000000000.0, migration_energy=0.6, drift_coefficient=10.0, cte_red=2.5e-10, recom_enchancement_factor=3000.0, decaimiento_concentracion=1e-09, activation_energy=1.01, gamma=8.0, ohm_resistence=11.5, pb_metal_insul=0.5, permitividad_relativa=20.0, I_0=0.0018, r_termica_percola=500.0, r_termica_no_percola=10.0, factor_generacion=2.0, recombination_energy=1.04)


In [ ]:
def PP_set(
    num_simulation: int,
    actual_state: np.ndarray,
    params: SimulationParameters,
    sim_ctes: SimulationConstants,
    CF_ranges: List[tuple],
    CF_creado: np.ndarray,
):
    
    # Pongo el nombre de la simulación y un salto de línea
    print(f"\nSimulacion {num_simulation + 1}")
    
    # Declaro todas las variables que voy a usar exclusivamente en la primera parte (PP) del set.

    # Rutas de guardado: 
    simulation_path = Path.cwd() / "Results" / f"simulation_{num_simulation + 1}"
    figures_path = simulation_path / "Figures"
    set_simulation_path = simulation_path / "set"

    # Crea las carpeta de la simualacion
    simulation_path.mkdir(parents=True, exist_ok=True)
    figures_path.mkdir(parents=True, exist_ok=True)
    set_simulation_path.mkdir(parents=True, exist_ok=True)

    # Inicializo vectores donde almaceno datos
    E_field_vector = np.zeros((actual_state.shape[0]), dtype=np.float64)
    
    sistema_percola = False
    num_max_vacantes = (params.device_size / params.atom_size) ** 2

    ocupacion_max_pp_set = 0.35
    num_max_vacantes = (params.device_size / params.atom_size) ** 2
    max_vancantes_pp_set = int(ocupacion_max_pp_set * num_max_vacantes)
    
    paso_temporal = params.total_simulation_time / params.num_pasos
    paso_potencial = params.voltaje_final_reset / params.num_pasos
    
    vector_ddp = np.arange(
        0.000, params.voltaje_final_reset + paso_potencial, paso_potencial
    )

    voltage_CF_creado = np.full(len(CF_ranges), 0.0)
    indice_filamento = 0
    
    # Defino las cabeceras de los archivos csv
    header_files = "Tiempo simulacion [s],Voltaje [V],Intensidad [A]"
    
    colunm_number = len(header_files.split(","))
    data_pp_set = np.zeros((params.num_pasos, colunm_number), dtype=np.float64)
    
    config_matrix_pp_set = np.zeros((int((params.num_pasos / params.paso_guardar)), params.x_size, params.y_size))
    
    temperatura = params.T_0

    print("\nComienza la primera parte del set")
    for k in range(0, params.num_pasos):
        # Si se llena el 90 del espacio de la matriz salto a otra simulación. Ponerlo aqui puede dar el problema de que nada mas empezar esté lleno y de error, pero eso NO debe pasar asi q no me preocupa.
        
        total_vacantes = np.sum(actual_state)
        
        if total_vacantes > int(0.9 * num_max_vacantes):
            raise exceptions.MaxVacantesException(k=k-1, voltage=vector_ddp[k-1])
        else:
            # Verifica si el sistema ha percolado
            if not sistema_percola:
                raise exceptions.NoPercolationException()

        # Actualizo el tiempo de simulación y el voltaje
        simulation_time = paso_temporal * k
        voltage = vector_ddp[k]

        _, CF_graph = CurrentSolver.Clean_state_matrix(actual_state)

        max_x, max_y = actual_state.shape
        filamentos = CurrentSolver.Clasificar_CF(CF_graph, max_x, max_y, CF_ranges)

        # Compruebo si hay filamentos nuevos
        if any(~CF_creado):  # mientras haya alguno sin romper
            filamentos_nuevos = [
                i
                for i, v in enumerate(CurrentSolver.Existe_filamentos(filamentos, len(CF_ranges)))
                if v and not CF_creado[i]
            ]

        for i in filamentos_nuevos:
            CF_creado[i] = True
            voltage_CF_creado[indice_filamento] = voltage
            indice_filamento += 1
            Representate.RepresentateState(
                actual_state,
                round(voltage, 3),
                str(simulation_path) + f"Figures/Filamento_{i + 1}_creado_set_{num_simulation + 1}.png",
            )
            print(f"El filamento {i + 1} se ha creado en el voltaje {voltage}")

        if voltage >= params.voltaje_final_set:
            # Verifica si el sistema ha percolado
            if not sistema_percola:
                raise exceptions.NoPercolationException()

            voltaje_max_set = vector_ddp[k-1]
            tiempo_pp_set = paso_temporal * (
                k - 1
            )  # Le quitamos un paso porque se ha superado el voltaje de ruptura

            print(
                "Voltaje final set",
                voltaje_max_set,
                " en el tiempo ",
                tiempo_pp_set,
                "\n",
            )

            # Elimino las filas sobrantes del array de datos y las lleno de nans para eliminarlas luego
            data_pp_set[k - 1 :] = np.nan  # Añadir valores nulos a partir de la fila k
            data_pp_set = data_pp_set[~np.isnan(data_pp_set).any(axis=1)]  # Eliminar filas con valores nulos
            break

        # Obtengo la corrriente, antes decido cual usar comprobando si ha percolado o no
        if Percolation.is_path(actual_state):
            # Si es la primera vez que percola, siste_percola será falso y entra aquí
            if sistema_percola is False:
                voltaje_percolacion = voltage  # Guardo el voltaje de percolación
                print(
                    "\nEl sistema ha percolado en la iteración: ", k,
                    " que corresponde con el voltaje: ",
                    voltaje_percolacion, " con una ocupación del: ",
                    (np.sum(actual_state) / (num_max_vacantes)) * 100, " %",
                )
                
                if voltaje_percolacion >= 0.75:
                    # Si el voltaje de percolación es demasiado alto no va a coincidir con los datos experimentales, y no merece la pena seguir con la simulación
                    raise exceptions.HighPercolationVoltageException(voltage_percola=voltaje_percolacion)
                
                Representate.RepresentateState(
                    actual_state,
                    round(voltaje_percolacion, 3),
                    str(simulation_path) + f"Figures/Percola_state_{num_simulation + 1}.png",
                )

                # Cambio la probabilidad de generación de vacantes para controlar la percolación
                sim_ctes.update_gamma(sim_ctes.gamma / sim_ctes.factor_generacion)
                print("Nueva gamma cuando percola el sistema: ", sim_ctes.gamma)

            sistema_percola = True

            exist_cf = CurrentSolver.Existe_filamentos(filamentos, len(CF_ranges))
            cf_clean_matrix = CurrentSolver.Eliminar_filamentos_incompletos(CF_graph, CF_ranges, exist_cf)

            # Si ha percolado uso la corriente de Ohm
            try:
                current, _ = CurrentSolver.OmhCurrent(
                    voltage, cf_clean_matrix, **asdict(sim_ctes)
                )
            except ZeroDivisionError:
                raise exceptions.NullResistanceException(simulation_path=simulation_path, voltage=voltage, num_simulation=num_simulation+1, actual_state=actual_state)

        else:
            sistema_percola = False
            
            mean_field = np.mean(E_field_vector).item()
            # Si no ha percolado uso la corriente de Poole-Frenkel
            current = CurrentSolver.Poole_Frenkel(temperatura, mean_field ,**asdict(sim_ctes)
            ) * (params.device_size)

        # Obtengo los valores del campo eléctrico y la temperatura
        # E_field = ElectricField.SimpleElectricField(voltage, params.device_size)
        temperatura = Temperature.Temperature_Joule(
            voltage, current, sistema_percola, params.T_0, **asdict(sim_ctes)
        )

        # Genero el vector campo eléctrico
        for i in range(0, actual_state.shape[0]):
            E_field_vector[i] = ElectricField.GapElectricField(
                voltage, i, actual_state, **sim_parmtrs[num_simulation]
            )

        # calcular las probabilidades por fila
        prob_generacion_fila = np.minimum(
            [
                Generation.Generate(
                    paso_temporal,
                    E_field_vector[i],
                    temperatura,
                    **asdict(sim_ctes),
                )
                for i in range(params.x_size)
            ],
            1,
        )
                            
        # Calculo la probabilidad de generación o recombinación para ello recorro toda la matriz
        for i in range(params.x_size):
            base_prob = prob_generacion_fila[i]
            for j in range(params.y_size):
                if actual_state[i, j] == 0:
                    if total_vacantes < max_vancantes_pp_set:
                        # Compruebo si tiene una vacate cerca
                        if Generation.vecinos_horizontales(actual_state, i, j):
                            prob_generacion = base_prob * 1.1
                        else:
                            prob_generacion = base_prob * 0.9
                    else:
                        prob_generacion = 0  # LO hago para que no se generen más vacantes y no se llene el sistema

                    random_number = np.random.rand()
                    if random_number < prob_generacion:
                        actual_state[i, j] = 1  # Generación de una vacante


        data_pp_set[k - 1] = np.array([simulation_time, voltage,current])

        # Guardo el estado actual CADA paso_guardar PASOS MONTECARLO
        if k % params.paso_guardar == 0:
            config_matrix_pp_set[int(k / params.paso_guardar) - 1] = actual_state


In [4]:
ruta_raiz = Path.cwd()
    
carpeta_results = ruta_raiz / "Results"

print("carpeta_results", carpeta_results)

carpeta_results c:\Users\Usuario\Documents\GitHub\RRAM_Simulation\Results
